# 바로핏 런지 · 공통 반환 규격 2.0 확장안
    사진의 구조를 따라 **MediaPipe → 운동별 규칙 → frame.result → set.completed**로 구성했습니다. 정자세1·오류4의 사진 위 스켈레톤 GIF, 각도 그래프, 카운트, 피드백과 JSON을 저장합니다.

    **호환성:** 받은 원본 스키마는 pushup/plank/bird_dog 전용이므로 이 노트북의 squat/lunge 결과는 원본에 그대로 통과하지 않습니다. 구조를 따르는 별도 확장 스키마를 함께 생성하고 검증합니다. 기존 운동 규격을 덮어쓰지 않으며 팀 합의 후 원본 스키마에 운동 분기를 추가해야 합니다.

    **현재와 회차 분리:** items/currentCodes는 현재 관측, lastRepetition은 최근 완료 회차입니다. 깊이 부족은 복귀가 끝난 회차에서 평가합니다. codes는 현재와 최근 회차 코드의 중복 없는 합집합입니다. 경고 없음과 판단 불가는 구분합니다.

    **동작 계수:** 충분히 서기 → 무릎125° 이하로 굽히기 → 서기. 규칙을 넘지 않은 얕은 시도는 세지 않습니다. 척추 중립·무릎 내측 붕괴·통증 여부는 이 측면 각도 엔진에서 판정하지 않습니다. 수치 기준은 검증 전 실험 초기값입니다. 실시간 엔진은 미래 프레임·원본 정상/오류 라벨을 사용하지 않습니다. 이 노트북의 실행과 GIF 생성은 오프라인입니다.

    각 셀을 1번부터 실행하세요. 시작할 때 서 있는 자세가 필요합니다. 원본 세트 JSON은 전체 영상 기준이고, GIF의 구간 횟수는 원본 이벤트를 선택 구간에 연결한 값입니다. 반복 재생 자체는 횟수를 올리지 않습니다.


## 1. 설치
새 Colab 런타임에서 실행하세요. 기존 환경의 import 충돌이 있으면 런타임을 재시작합니다.


In [ ]:
import sys,subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','mediapipe==0.10.32','opencv-python-headless>=4.10,<5','numpy>=1.26,<3','pandas>=2.2,<3','pillow','matplotlib','jsonschema>=4.23,<5'],check=True)


## 2. Drive 경로와 실행 설정
SIDE는 기본 auto입니다. 런지는 앞다리를 확인한 뒤 SIDE_OVERRIDES로 left/right를 지정하는 것을 권장합니다. 자동 선택은 초기 가시성 기반이며 앞다리를 보장하지 않습니다. 기준값은 5번 셀 DEFAULTS에서 확인합니다.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json,hashlib,urllib.request,unicodedata,math,copy
import numpy as np,pandas as pd,cv2,mediapipe as mp
from PIL import Image,ImageDraw,ImageFont
from IPython.display import display,Image as DisplayImage
import matplotlib.pyplot as plt

drive.mount('/content/drive')
EXERCISE='lunge'
DATA_ROOT=Path('/content/drive/MyDrive/REHAB24-6_Ex5')
OUTPUT_ROOT=Path('/content/drive/MyDrive/Barofit')/(EXERCISE+'_v2_results')
RUN_DIR=OUTPUT_ROOT/datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True,exist_ok=False)
CACHE_DIR=OUTPUT_ROOT/'pose_cache';CACHE_DIR.mkdir(exist_ok=True)
SAMPLE_FPS=10
GIF_FPS=5
GIF_MAX_SECONDS=20
FRAME_INDEX_BASE=0
SIDE='auto'
SIDE_OVERRIDES={}
THRESHOLD_OVERRIDES={}
MANUAL_CASES=[]
# 직접 지정 예: path는 DATA_ROOT 아래 상대 경로 또는 절대 경로입니다.
# MANUAL_CASES=[dict(path='normal.mp4',start=0,end=6,label='good'),
#               dict(path='bad.mp4',start=0,end=6,label='bad'), ...]
# 런지 앞다리 선택 예: SIDE_OVERRIDES={'영상파일명.mp4':'left'}
# 복귀 시 무릎이 155도까지 펴지지 않는 영상은 측정값 확인 후 기준을 조절하세요.
# THRESHOLD_OVERRIDES={'stand':145.}
if not DATA_ROOT.is_dir():raise FileNotFoundError(f'2번 셀 DATA_ROOT를 확인하세요: {DATA_ROOT}')
print('운동:',EXERCISE,'\n결과:',RUN_DIR)


## 3. 정자세 1개·오류 자세 4개 선택
런지는 REHAB24-6 Ex5 CSV의 correctness=1/0을 사용합니다. 스쿼트 및 자동 연결 실패 시 파일번호와 구간을 직접 입력합니다. 서로 다른 4명이나 4오류 유형을 보장하지 않습니다. 원본 미리보기에서 운동과 대상자를 확인하세요. MANUAL_CASES로 같은 원본의 다른 구간도 지정할 수 있습니다.


In [ ]:
videos=sorted(p for p in DATA_ROOT.rglob('*') if p.suffix.lower() in {'.mp4','.avi','.mov','.mkv'} and CACHE_DIR not in p.parents)
if not videos:raise FileNotFoundError('데이터 폴더 아래에서 영상을 찾지 못했습니다.')
CLIPS=[]
if MANUAL_CASES:
    for c in MANUAL_CASES:
        p=Path(c['path']);p=p if p.is_absolute() else DATA_ROOT/p
        CLIPS.append(dict(path=str(p),start=float(c.get('start',0)),end=c.get('end'),label=c['label'],labelSource='user_selected'))
elif EXERCISE=='lunge':
    label_paths=sorted(DATA_ROOT.rglob('*Segmentation*.csv'))
    if label_paths:
        labels=pd.concat([pd.read_csv(p,sep=';',encoding='utf-8-sig') for p in label_paths],ignore_index=True)
        for col in ['exercise_id','correctness','first_frame','last_frame','person_id']:
            labels[col]=pd.to_numeric(labels[col],errors='coerce')
        labels=labels[labels.exercise_id==5].dropna(subset=['video_id','correctness','first_frame','last_frame'])
        keys=['video_id','repetition_number'] if 'repetition_number' in labels else ['video_id','first_frame','last_frame']
        if (labels.groupby(keys)[['first_frame','last_frame','correctness']].nunique()>1).any().any():raise ValueError('중복 라벨 값이 서로 다릅니다.')
        labels=labels.drop_duplicates(keys)
        candidates=[]
        for _,r in labels.iterrows():
            orientation=str(r.get('cam17_orientation','')).strip().lower()
            camera=17 if orientation in ('side','profile') else 18 if orientation=='front' else None
            if camera is None:continue
            extra=pd.to_numeric(r.get(f'extra_person_in_cam{camera}',0),errors='coerce')
            if pd.isna(extra) or extra>=3:continue
            found=[p for p in videos if p.name.startswith(str(r.video_id)+f'-Camera{camera}-')]
            if len(found)!=1:continue
            if r.correctness not in (0,1):continue
            candidates.append(dict(path=str(found[0]),start=(int(r.first_frame)-FRAME_INDEX_BASE)/30,end=(int(r.last_frame)-FRAME_INDEX_BASE+1)/30,label='good' if r.correctness==1 else 'bad',person=r.person_id,labelSource='REHAB24-6 CSV correctness'))
        good=[c for c in candidates if c['label']=='good'];bad=[c for c in candidates if c['label']=='bad']
        chosen=[];seen=set()
        for c in bad:
            if c['person'] not in seen:chosen.append(c);seen.add(c['person'])
            if len(chosen)==4:break
        for c in bad:
            if len(chosen)==4:break
            if c not in chosen:chosen.append(c)
        if good and len(chosen)==4:CLIPS=[good[0]]+chosen
if not CLIPS:
    print('각 파일의 운동과 정상/오류 여부를 확인하고 번호를 선택하세요. 파일명만으로 라벨을 추정하지 않습니다.')
    for i,p in enumerate(videos):print(f'{i}: {p.relative_to(DATA_ROOT)}')
    print('입력 예: 0:0:8, 1:0:8, 2:0:8, 3:0:8, 4:0:8')
    print('형식은 파일번호:시작초:끝초입니다. 파일번호만 쓰면 전체 영상입니다.')
    tokens=input('정자세 1개, 오류 4개 순서로 입력: ').split(',')
    if len(tokens)!=5:raise ValueError('구간 5개를 입력하세요.')
    for i,token in enumerate(tokens):
        parts=token.strip().split(':')
        if len(parts) not in (1,3):raise ValueError('파일번호 또는 파일번호:시작초:끝초 형식을 사용하세요.')
        idx=int(parts[0])
        if not 0<=idx<len(videos):raise ValueError('영상 번호 범위 오류')
        CLIPS.append(dict(path=str(videos[idx]),start=float(parts[1]) if len(parts)==3 else 0.,end=float(parts[2]) if len(parts)==3 else None,label='good' if i==0 else 'bad',labelSource='user_selected'))
if len(CLIPS)!=5 or [c['label'] for c in CLIPS]!=['good','bad','bad','bad','bad']:raise ValueError('정자세1·오류4 순서여야 합니다.')
for i,c in enumerate(CLIPS):
    c['case']=f'case_{i}'
    cap=cv2.VideoCapture(c['path'])
    try:
        if not cap.isOpened():raise FileNotFoundError(c['path'])
        fps=cap.get(cv2.CAP_PROP_FPS);total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps<=0 or total<2:raise ValueError('영상 FPS/프레임 정보 오류')
        duration=total/fps
        c['end']=duration if c['end'] is None else float(c['end'])
        if not 0<=c['start']<c['end']<=duration+.05:raise ValueError(f'구간 범위 오류: {c}')
        c['end']=min(c['end'],duration)
        if c['labelSource'].startswith('REHAB') and abs(fps-30)>.1:raise ValueError('CSV 라벨은 원본30FPS 영상이 필요합니다.')
        cap.set(cv2.CAP_PROP_POS_MSEC,1000*(c['start']+c['end'])/2)
        ok,bgr=cap.read()
        if not ok:raise RuntimeError('미리보기 읽기 실패')
        img=Image.fromarray(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB));img.thumbnail((480,320))
        print(c['case'],c['label'],Path(c['path']).name,c['start'],c['end']);display(img)
    finally:cap.release()
pd.DataFrame(CLIPS).to_csv(RUN_DIR/'selected_cases.csv',index=False,encoding='utf-8-sig')
print('라벨은 비교용 원본/사용자 라벨이며, 엔진이 정답으로 학습하거나 피드백을 강제하지 않습니다.')


## 4. 원본 영상 관절 추출과 캐시
원본 시간과 FPS를 보존합니다. 같은 영상은 한 번만 추출합니다. 모델 추론에는 원본 영상 전체를 사용하지만 엔진은 시간 순서대로만 입력받습니다. 인물 한 명의 측면 전신 영상용입니다.


In [ ]:
MODEL=Path('/content/models/pose_landmarker_full.task');MODEL.parent.mkdir(parents=True,exist_ok=True)
if not MODEL.exists():
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task',MODEL)
MODEL_HASH=hashlib.sha256(MODEL.read_bytes()).hexdigest()
def extract_pose(path):
    p=Path(path)
    key=hashlib.sha256(f'{p.resolve()}|{p.stat().st_size}|{p.stat().st_mtime_ns}|{MODEL_HASH}|{SAMPLE_FPS}|{mp.__version__}'.encode()).hexdigest()
    cache=CACHE_DIR/(key+'.npz')
    if cache.exists():
        with np.load(cache,allow_pickle=False) as f:return {k:f[k] for k in f.files}
    options=mp.tasks.vision.PoseLandmarkerOptions(base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL)),running_mode=mp.tasks.vision.RunningMode.VIDEO,num_poses=1,min_pose_detection_confidence=.5,min_pose_presence_confidence=.5,min_tracking_confidence=.5)
    cap=cv2.VideoCapture(path);poses=[];indices=[];times=[];dims=[];i=0;next_t=0.;last_ms=-1
    try:
        fps=cap.get(cv2.CAP_PROP_FPS);total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps<=0:raise ValueError('FPS 오류')
        with mp.tasks.vision.PoseLandmarker.create_from_options(options) as model:
            while True:
                ok,bgr=cap.read()
                if not ok:break
                t=i/fps
                if t+1e-6>=next_t:
                    h,w=bgr.shape[:2]
                    scale=min(1.,960/max(w,h));bgr=cv2.resize(bgr,(round(w*scale),round(h*scale)))
                    rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
                    timestamp=max(last_ms+1,round(t*1000));last_ms=timestamp
                    result=model.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb),timestamp)
                    pose=np.full((33,5),np.nan,dtype=float)
                    if result.pose_landmarks:
                        pose=np.array([[q.x,q.y,q.z,q.visibility,q.presence] for q in result.pose_landmarks[0]],dtype=float)
                    poses.append(pose);indices.append(i);times.append(t);dims.append([rgb.shape[1],rgb.shape[0]])
                    next_t+=1/SAMPLE_FPS
                    if len(times)%100==0:print(Path(path).name,f'{t:.1f}/{total/fps:.1f}초',flush=True)
                i+=1
        if not poses or i<total*.95:raise RuntimeError('원본 영상 디코딩이 불완전합니다.')
    finally:cap.release()
    arrays=dict(poses=np.array(poses),indices=np.array(indices),times=np.array(times),dims=np.array(dims),fps=np.array(fps))
    np.savez_compressed(cache,**arrays)
    return arrays
DATA={}
for path in dict.fromkeys(c['path'] for c in CLIPS):
    print('관절 추출:',Path(path).name,flush=True)
    DATA[path]=dict(arrays=extract_pose(path))


## 5. 공통 카운트·피드백 엔진
무릎 각도와 화면 수직 대비 몸통 기울기를 측정합니다. 지수 평활화·경계 유지·오류 지속 시간을 사용합니다. 인식 실패나 0.35초 초과 간격에서 미완료 동작을 끊습니다. 정상 복귀 후 다음 동작부터 재개합니다. 상체 항목은 동작 중 기울기 경고 지속 여부, 깊이는 완료 회차의 최저 무릎 각도로 평가합니다. 관절 인식이 부족한 항목은 unknown입니다. 모든 계산값은 실험용이며 최적 자세의 보편적 기준이 아닙니다.


In [ ]:
import math,copy
import numpy as np

SIDES={'left':(11,13,15,23,25,27),'right':(12,14,16,24,26,28)}
DEFAULTS={'squat':dict(stand=155.,bend=125.,depth=110.,trunk=45.),'lunge':dict(stand=150.,bend=125.,depth=110.,trunk=35.)}

def angle(a,b,c):
    u=a-b;v=c-b;den=np.linalg.norm(u)*np.linalg.norm(v)
    return None if den<1e-8 else float(np.degrees(np.arccos(np.clip(np.dot(u,v)/den,-1,1))))

def measurements(pose,width,height,quality=.5):
    result={}
    for side,(s,e,w,h,k,a) in SIDES.items():
        def valid(ids):
            if pose is None:return False
            q=pose[list(ids)]
            return bool(np.isfinite(q).all() and (q[:,3:]>=quality).all() and ((q[:,:2]>=0)&(q[:,:2]<=1)).all())
        xy=None if pose is None else pose[:,:2]*[width,height]
        knee=angle(xy[h],xy[k],xy[a]) if valid((h,k,a)) else None
        hip=angle(xy[s],xy[h],xy[k]) if valid((s,h,k)) else None
        elbow=angle(xy[s],xy[e],xy[w]) if valid((s,e,w)) else None
        trunk=float(np.degrees(np.arctan2(abs(xy[h,0]-xy[s,0]),xy[h,1]-xy[s,1]))) if valid((s,h)) and np.linalg.norm(xy[h]-xy[s])>1e-8 else None
        result[side]=dict(kneeDeg=knee,hipDeg=hip,elbowDeg=elbow,trunkDeg=trunk)
    return result

class BarofitEngine:
    VERSION='experimental-lower-body-2.0.0'
    def __init__(self,exercise,side='auto',overrides=None):
        if exercise not in DEFAULTS or side not in ('auto','left','right'):raise ValueError('운동 또는 분석측 설정 오류')
        self.exercise=exercise;self.config={**DEFAULTS[exercise],**(overrides or {})}
        if not 0<self.config['depth']<=self.config['bend']<self.config['stand']<=180:raise ValueError('각도 기준 순서를 확인하세요.')
        self.side=None if side=='auto' else side;self.scores={'left':0.,'right':0.};self.select_started=None
        self.last_t=None;self.count=0;self.last_rep=None;self.reps=[];self.finished=None;self.interrupted=0
        self.totals={k:dict(compliant=0,warning=0,unknown=0,notApplicable=0) for k in ('depth','trunk')}
        self.observed=0.;self.gaps=0.;self.reset_motion()
    def reset_motion(self):
        self.phase='WAIT_STANDING';self.pending=None;self.pending_since=None;self.active=None;self.lean_since=None;self.smooth={}
    def feedback(self,codes):
        messages={'DEPTH_SHALLOW':'무릎을 조금 더 굽혀 몸을 낮춰보세요.','TRUNK_LEAN':'상체가 과하게 숙여지지 않도록 조절하세요.'}
        return [messages[c] for c in codes if c in messages]
    def frame(self,timestamp,raw,ignored=False,reason=None,tracking='tracking_unavailable',items=None,current=None,events=None,m=None):
        recent=None if ignored else copy.deepcopy(self.last_rep)
        current=[] if current is None or ignored else list(current)
        codes=list(dict.fromkeys(current+([] if recent is None else recent['codes'])))
        return dict(schemaVersion='2.0',type='frame.result',exercise=self.exercise,engineVersion=self.VERSION,timestampMs=float(timestamp),ignored=ignored,ignoredReason=reason,tracking=tracking,side=self.side,count=self.count,holdSeconds=None,observedSeconds=None,measurementBasis=dict(count='completed_threshold_cycle',holdSeconds=None,observedSeconds=None),phase=self.phase,items={} if ignored else (items or {'depth':'tracking_unavailable','trunk':'tracking_unavailable'}),codes=[] if ignored else codes,currentCodes=current,lastRepetition=recent,metrics={'selected':m or {},'sides':copy.deepcopy(raw)},events=[] if ignored else (events or []),details=dict(experimental=True,thresholdBasis='configurable_unvalidated_side_view_thresholds',thresholds=dict(self.config),interruptedAttempts=self.interrupted,feedback=self.feedback(codes) if not ignored else [],currentFeedback=self.feedback(current),lastRepetitionFeedback=self.feedback(recent['codes']) if recent else [],diagnostics=dict(observedSeconds=self.observed,gapSeconds=self.gaps)))
    def step(self,timestamp,raw):
        t=float(timestamp)
        if not math.isfinite(t) or t<0:raise ValueError('timestampMs는 유한한 0 이상 숫자여야 합니다.')
        if self.finished is not None:return self.frame(t,raw,True,'finished')
        if self.last_t is not None and t<=self.last_t:return self.frame(t,raw,True,'non_monotonic_timestamp')
        dt=0. if self.last_t is None else (t-self.last_t)/1000;self.last_t=t
        events=[]
        if dt>.35:
            self.gaps+=dt
            if self.active is not None:self.interrupted+=1;events.append(dict(type='repetition.interrupted',reason='sample_gap'))
            self.reset_motion()
        if self.side is None:
            if self.select_started is None:self.select_started=t
            for side in SIDES:
                if raw.get(side,{}).get('kneeDeg') is not None:self.scores[side]+=1
            if t-self.select_started>=800 and max(self.scores.values())>=4:self.side=max(self.scores,key=self.scores.get)
            else:return self.frame(t,raw,tracking='selecting_side',items={'depth':'pending_feedback','trunk':'pending_feedback'})
        src=raw.get(self.side,{})
        if src.get('kneeDeg') is None:
            if self.active is not None:self.interrupted+=1;events.append(dict(type='repetition.interrupted',reason='tracking_lost'))
            self.reset_motion();return self.frame(t,raw,events=events)
        m={}
        for key,value in src.items():
            if value is None:self.smooth.pop(key,None);m[key]=None
            else:
                old=self.smooth.get(key,float(value));alpha=1-math.exp(-max(dt,.001)/.10)
                m[key]=old+alpha*(float(value)-old);self.smooth[key]=m[key]
        if 0<dt<=.35:self.observed+=dt
        knee=m['kneeDeg'];trunk=m.get('trunkDeg')
        if self.phase=='READY' and knee<self.config['stand']-3 and self.active is None:
            self.active=dict(start=t,minimum=knee,trough=t,seen=0,validTrunk=0,trunkWarning=False,maxTrunk=None)
        if self.active is not None:
            a=self.active;a['seen']+=1
            if knee<a['minimum']:a['minimum']=knee;a['trough']=t
            if trunk is not None:
                a['validTrunk']+=1;a['maxTrunk']=trunk if a['maxTrunk'] is None else max(a['maxTrunk'],trunk)
        leaning=trunk is not None and knee<self.config['stand']-5 and trunk>self.config['trunk']
        if leaning:
            if self.lean_since is None:self.lean_since=t
        else:self.lean_since=None
        lean_warning=self.lean_since is not None and t-self.lean_since>=300
        if lean_warning and self.active is not None:self.active['trunkWarning']=True
        observed='up' if knee>=self.config['stand'] else 'down' if knee<=self.config['bend'] else None
        if observed is None:self.pending=None;self.pending_since=None
        else:
            if self.pending!=observed:self.pending=observed;self.pending_since=t
            if t-self.pending_since>=100:
                if observed=='up' and self.phase=='WAIT_STANDING' and trunk is not None and trunk<60:self.phase='READY'
                elif observed=='down' and self.phase=='READY':self.phase='LOWERED'
                elif observed=='up' and self.phase=='LOWERED':
                    a=self.active
                    if a is not None and t-a['start']>=500:
                        self.count+=1
                        depth='warning' if a['minimum']>self.config['depth'] else 'within_reference'
                        torso='unknown' if a['validTrunk']/max(1,a['seen'])<.7 else 'warning' if a['trunkWarning'] else 'within_reference'
                        repitems=dict(depth=depth,trunk=torso)
                        repcodes=(['DEPTH_SHALLOW'] if depth=='warning' else [])+(['TRUNK_LEAN'] if torso=='warning' else [])
                        self.last_rep=dict(number=self.count,timestampMs=t,items=repitems,codes=repcodes,metrics=dict(startTimestampMs=a['start'],troughTimestampMs=a['trough'],minimumKneeDeg=a['minimum'],maximumTrunkDeg=a['maxTrunk']))
                        self.reps.append(copy.deepcopy(self.last_rep))
                        for key,status in repitems.items():self.totals[key][{'within_reference':'compliant','warning':'warning','unknown':'unknown'}[status]]+=1
                        events.append(dict(type='repetition.completed',number=self.count))
                    else:
                        self.interrupted+=1;events.append(dict(type='repetition.interrupted',reason='too_short'))
                    self.phase='READY';self.active=None
                elif observed=='up' and self.phase=='READY':self.active=None
        items=dict(depth='pending_feedback' if self.active is not None else 'not_applicable',trunk='unknown' if trunk is None else 'warning' if lean_warning else 'within_reference')
        return self.frame(t,raw,tracking='tracking',items=items,current=['TRUNK_LEAN'] if lean_warning else [],events=events,m=m)
    def finish(self):
        if self.finished is not None:return copy.deepcopy(self.finished)
        if self.active is not None:self.interrupted+=1;self.reset_motion()
        items={}
        for key,values in self.totals.items():
            evaluated=values['compliant']+values['warning']
            items[key]=dict(aggregationBasis='repetition',unit='repetitions',**values,evaluated=evaluated,compliancePct=100*values['compliant']/evaluated if evaluated else None)
        summary=dict(schemaVersion='2.0',exercise=self.exercise,engineVersion=self.VERSION,count=self.count,holdSeconds=None,observedSeconds=None,gapSeconds=None,items=items,details=dict(repetitions=copy.deepcopy(self.reps),interruptedAttempts=self.interrupted,experimental=True,thresholds=dict(self.config),diagnostics=dict(observedSeconds=self.observed,gapSeconds=self.gaps)))
        self.finished=dict(schemaVersion='2.0',type='set.completed',exercise=self.exercise,engineVersion=self.VERSION,timestampMs=self.last_t,summary=summary)
        return copy.deepcopy(self.finished)


## 6. 확장 JSON Schema와 집계 검증
두 운동의 count는 completed_threshold_cycle입니다. 유지 시간 필드는 null, 항목별 집계는 repetition/repetitions입니다. evaluated=compliant+warning, 준수율 분모에서 unknown/notApplicable을 제외합니다. 계산식은 별도 코드로 검증합니다.


In [ ]:
from jsonschema import Draft202012Validator
STATUS=['within_reference','warning','tracking_unavailable','pending_feedback','body_unavailable','body_deviation','interrupted','unknown','not_applicable','disabled','not_supported']
def obj(properties,required=None):
    return dict(type='object',properties=properties,required=list(properties) if required is None else required,additionalProperties=False)
def nullable(kind,minimum=0):
    return dict(type=[kind,'null'],minimum=minimum)
ITEMS=dict(type='object',additionalProperties=dict(enum=STATUS))
METRICS=dict(type='object')
CODES=dict(type='array',items=dict(type='string'))
REP=obj(dict(number=nullable('integer',1),timestampMs=nullable('number'),items=ITEMS,codes=CODES,metrics=METRICS))
BASIS=dict(count='completed_threshold_cycle',holdSeconds=None,observedSeconds=None)
FRAME_SCHEMA=obj(dict(schemaVersion=dict(const='2.0'),type=dict(const='frame.result'),exercise=dict(enum=['squat','lunge']),engineVersion=dict(type='string'),timestampMs=dict(type='number',minimum=0),ignored=dict(type='boolean'),ignoredReason=dict(enum=[None,'finished','non_monotonic_timestamp']),tracking=dict(enum=['tracking','selecting_side','tracking_unavailable']),side=dict(enum=[None,'left','right']),count=dict(type='integer',minimum=0),holdSeconds=dict(type='null'),observedSeconds=dict(type='null'),measurementBasis=dict(const=BASIS),phase=dict(type=['string','null']),items=ITEMS,codes=CODES,currentCodes=CODES,lastRepetition=dict(oneOf=[dict(type='null'),REP]),metrics=METRICS,events=dict(type='array',items=dict(type='object',required=['type'],properties=dict(type=dict(type='string')))),details=METRICS))
FRAME_SCHEMA['$schema']='https://json-schema.org/draft/2020-12/schema'
FRAME_SCHEMA['title']='Proposed Barofit v2 extension: squat and lunge frames'
FRAME_SCHEMA['allOf']=[
    {'if':{'properties':{'ignored':{'const':True}}},'then':{'properties':{'ignoredReason':{'enum':['finished','non_monotonic_timestamp']},'items':{'maxProperties':0},'codes':{'maxItems':0},'currentCodes':{'maxItems':0},'lastRepetition':{'type':'null'},'events':{'maxItems':0}}},'else':{'properties':{'ignoredReason':{'type':'null'}}}}
]
AGG=obj(dict(aggregationBasis=dict(const='repetition'),unit=dict(const='repetitions'),compliant=dict(type='integer',minimum=0),warning=dict(type='integer',minimum=0),unknown=dict(type='integer',minimum=0),notApplicable=dict(type='integer',minimum=0),evaluated=dict(type='integer',minimum=0),compliancePct=dict(type=['number','null'],minimum=0,maximum=100)))
AGG['allOf']=[{'if':{'properties':{'evaluated':{'const':0}}},'then':{'properties':{'compliancePct':{'type':'null'}}},'else':{'properties':{'compliancePct':{'type':'number'}}}}]
SUMMARY=obj(dict(schemaVersion=dict(const='2.0'),exercise=dict(enum=['squat','lunge']),engineVersion=dict(type='string'),count=dict(type='integer',minimum=0),holdSeconds=dict(type='null'),observedSeconds=dict(type='null'),gapSeconds=dict(type='null'),items=dict(type='object',additionalProperties=AGG),details=METRICS))
SET_SCHEMA=obj(dict(schemaVersion=dict(const='2.0'),type=dict(const='set.completed'),exercise=dict(enum=['squat','lunge']),engineVersion=dict(type='string'),timestampMs=nullable('number'),summary=SUMMARY))
SET_SCHEMA['$schema']=FRAME_SCHEMA['$schema']
SET_SCHEMA['title']='Proposed Barofit v2 extension: squat and lunge set summary'
SET_SCHEMA['oneOf']=[dict(properties=dict(exercise=dict(const=e),summary=dict(properties=dict(exercise=dict(const=e))))) for e in ['squat','lunge']]
Draft202012Validator.check_schema(FRAME_SCHEMA);Draft202012Validator.check_schema(SET_SCHEMA)
FRAME_VALIDATOR=Draft202012Validator(FRAME_SCHEMA)
SET_VALIDATOR=Draft202012Validator(SET_SCHEMA)
def validate_result(result):
    json.dumps(result,allow_nan=False)
    if result['type']=='frame.result':FRAME_VALIDATOR.validate(result)
    else:
        SET_VALIDATOR.validate(result)
        assert result['exercise']==result['summary']['exercise']
        assert result['engineVersion']==result['summary']['engineVersion']
        for item in result['summary']['items'].values():
            assert item['evaluated']==item['compliant']+item['warning']
            expected=100*item['compliant']/item['evaluated'] if item['evaluated'] else None
            assert item['compliancePct']==expected
            assert sum(item[k] for k in ['compliant','warning','unknown','notApplicable'])==result['summary']['count']
for name,schema in [('frame_result_extension.schema.json',FRAME_SCHEMA),('set_completed_extension.schema.json',SET_SCHEMA)]:
    (RUN_DIR/name).write_text(json.dumps(schema,ensure_ascii=False,indent=2),encoding='utf-8')
print('확장 스키마 생성 완료. 기존 pushup/plank/bird_dog 전용 스키마는 수정하지 않습니다.')


## 7. 분석 실행 · 프레임 JSONL / 세트 종료 JSON
현재 안내와 최근 회차 평가는 같은 엔진 결과에서 읽습니다. 원본 전체 세트를 저장하며 중복 파일명은 경로 해시로 구분합니다. API 키나 LLM 호출은 없습니다.


In [ ]:
for path,data in DATA.items():
    a=data['arrays'];engine=BarofitEngine(EXERCISE,SIDE_OVERRIDES.get(Path(path).name,SIDE),THRESHOLD_OVERRIDES)
    results=[];records=[];target=RUN_DIR/(Path(path).stem+'_'+hashlib.sha1(path.encode()).hexdigest()[:8])
    with target.with_suffix('.frames.jsonl').open('w',encoding='utf-8') as f:
        for i,(pose,t,dims) in enumerate(zip(a['poses'],a['times'],a['dims'])):
            raw=measurements(pose,*dims)
            r=engine.step(float(t)*1000,raw);validate_result(r)
            results.append(r);f.write(json.dumps(r,ensure_ascii=False,allow_nan=False)+'\n')
            m=r['metrics']['selected'];row=dict(time_s=float(t),source_frame=int(a['indices'][i]),count=r['count'],tracking=r['tracking'],side=r['side'],phase=r['phase'],currentCodes='|'.join(r['currentCodes']))
            for s in SIDES:
                for key,value in raw[s].items():row[s+'_'+key]=value
            row.update(m);records.append(row)
    completed=engine.finish();validate_result(completed)
    target.with_suffix('.set.json').write_text(json.dumps(completed,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    table=pd.DataFrame(records);table.to_csv(target.with_suffix('.csv'),index=False,encoding='utf-8-sig')
    data.update(results=results,completed=completed,table=table)
    print(Path(path).name,'완료 횟수:',completed['summary']['count'],'분석측:',engine.side)
    display(pd.DataFrame(completed['summary']['items']).T)
    if completed['summary']['count']==0:print('0회 진단: 시작 시 서기·측면 전신·분석측·무릎 굽힘/복귀 기준을 확인하세요. 판단 불가를 정상으로 바꾸지 않습니다.')
print('프레임 JSONL과 세트 종료 JSON은 선택 구간이 아닌 원본 영상 전체에 대한 결과입니다.')


## 8. 사진 위 스켈레톤 비교 GIF 4개
왼쪽은 정자세 라벨 구간, 오른쪽은 오류 라벨 구간입니다. 원본 영상에서 검출한 동작의 최저점으로 구간을 연결하고 복귀까지 최대1.5초 포함합니다. GIF는 기본 앞20초 미리보기이며 2번 셀 GIF_MAX_SECONDS로 조정합니다. GIF의 현재 관측과 최근 회차 평가는 분리 표시되고, 전체 구간 요약은 오프라인 결과입니다.


In [ ]:
font_path=Path('/content/NotoSansKR.ttf')
if not font_path.exists():urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/notosanskr/NotoSansKR%5Bwght%5D.ttf',font_path)
FONT=ImageFont.truetype(str(font_path),16)
EDGES=[(11,12),(11,13),(13,15),(12,14),(14,16),(11,23),(12,24),(23,24),(23,25),(25,27),(24,26),(26,28)]
GUIDANCE={'DEPTH_SHALLOW':'무릎을 조금 더 굽혀 몸을 낮춰보세요.','TRUNK_LEAN':'상체가 과하게 숙여지지 않도록 조절하세요.'}
for c in CLIPS:
    data=DATA[c['path']];a=data['arrays']
    reps=data['completed']['summary']['details']['repetitions']
    c['matched']=[r for r in reps if c['start']*1000<=r['metrics']['troughTimestampMs']<c['end']*1000]
    c['render_start']=max(0,c['start']-.4)
    c['render_end']=min(float(a['times'][-1]),max([c['end']]+[min(c['end']+1.5,r['timestampMs']/1000+.2) for r in c['matched']]))
    c['ids']=np.flatnonzero((a['times']>=c['render_start'])&(a['times']<=c['render_end']))
    if len(c['ids'])<2:raise ValueError('재생 구간이 너무 짧습니다.')
    c['counts']=np.array([sum(r['timestampMs']<=t*1000 for r in c['matched']) for t in a['times']],dtype=int)
    codes=list(dict.fromkeys(code for rep in c['matched'] for code in rep['codes']))
    c['summaryFeedback']=[GUIDANCE[k] for k in codes]
    if not c['matched']:c['summaryFeedback']=['완료 회차 미검출: 시작 자세와 굽힘·복귀 기준을 확인하세요.']
    elif not c['summaryFeedback']:
        unknown=any('unknown' in r['items'].values() for r in c['matched'])
        c['summaryFeedback']=['일부 항목은 관절 인식 부족으로 판단을 보류했습니다.' if unknown else '측정한 깊이·상체 기울기에서 경고가 없습니다.']
        if c['label']=='bad':c['summaryFeedback'].append('원본 오류 라벨의 원인을 현재 규칙이 모두 설명하지는 않습니다.')

def wrapped(draw,messages,y,limit=5):
    lines=[]
    for message in messages:
        buf=''
        for ch in message:
            if draw.textlength(buf+ch,font=FONT)>470:lines.append(buf);buf=ch
            else:buf+=ch
        if buf:lines.append(buf)
    for i,line in enumerate(lines[:limit]):draw.text((12,y+i*23),line,font=FONT,fill='white')

def panel(c,cap,elapsed):
    data=DATA[c['path']];a=data['arrays'];target=min(c['render_start']+elapsed,c['render_end'])
    past=c['ids'][a['times'][c['ids']]<=target+1e-6]
    n=int(past[-1] if len(past) else c['ids'][0])
    r=data['results'][n];p=a['poses'][n]
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(a['indices'][n]));ok,bgr=cap.read()
    if not ok:raise RuntimeError('GIF용 원본 프레임 읽기 실패')
    photo=Image.fromarray(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB));photo.thumbnail((480,340))
    canvas=Image.new('RGB',(500,800),'#101827');x0=(500-photo.width)//2;y0=125+(340-photo.height)//2;canvas.paste(photo,(x0,y0));draw=ImageDraw.Draw(canvas)
    def valid(j):return bool(np.isfinite(p[j]).all() and (p[j,3:]>=.5).all() and ((p[j,:2]>=0)&(p[j,:2]<=1)).all())
    def xy(j):return (x0+p[j,0]*photo.width,y0+p[j,1]*photo.height)
    for u,v in EDGES:
        if valid(u) and valid(v):draw.line([xy(u),xy(v)],fill='#5cdbff',width=3)
    for j in sorted({x for edge in EDGES for x in edge}):
        if valid(j):
            x,y=xy(j);draw.ellipse((x-3,y-3,x+3,y+3),fill='#ffe66d')
    m=r['metrics']['selected']
    def fmt(k):return '?' if m.get(k) is None else f'{m[k]:.0f}'
    draw.text((12,8),f"{EXERCISE} | {c['case']} | 라벨: {c['label']}",font=FONT,fill='white')
    draw.text((12,35),f"구간 {c['counts'][n]}회 / 원본 누적 {r['count']}회",font=FONT,fill='#a6f0b0')
    draw.text((12,62),f"{a['times'][n]:.2f}s | {r['side']} | {r['phase']}",font=FONT,fill='white')
    draw.text((12,89),f"무릎 {fmt('kneeDeg')}° / 몸통 기울기 {fmt('trunkDeg')}°",font=FONT,fill='white')
    draw.text((12,480),'현재 관측 · items / currentCodes',font=FONT,fill='#ffdf99')
    if r['tracking']!='tracking':current=['전신과 무릎이 보이도록 카메라를 조절하세요.']
    else:current=r['details']['currentFeedback'] or ['현재 경고 없음. 깊이는 회차 완료 후 평가합니다.']
    wrapped(draw,current,508,3)
    recent=r['lastRepetition']
    if recent and recent['number'] not in {x['number'] for x in c['matched']}:recent=None
    draw.text((12,585),'최근 완료 회차 · lastRepetition',font=FONT,fill='#ffdf99')
    if recent:
        last=[GUIDANCE[x] for x in recent['codes']] or ['측정 항목 경고 없음.']
        if 'unknown' in recent['items'].values():last.append('일부 항목 판단 보류.')
    else:last=['완료한 구간 동작이 아직 없습니다.']
    wrapped(draw,last,612,3)
    draw.text((12,695),'구간 전체 요약 · 오프라인',font=FONT,fill='#ffdf99')
    wrapped(draw,c['summaryFeedback'],722,3)
    return canvas

for bad in CLIPS[1:]:
    pair=[CLIPS[0],bad];caps=[cv2.VideoCapture(c['path']) for c in pair]
    full_duration=max(c['render_end']-c['render_start'] for c in pair)
    duration=min(GIF_MAX_SECONDS,full_duration)
    elapsed=np.unique(np.r_[np.arange(0,duration,1/GIF_FPS),duration]);frames=[]
    try:
        for t in elapsed:
            img=Image.new('RGB',(1010,800),'white')
            for i,(c,cap) in enumerate(zip(pair,caps)):img.paste(panel(c,cap,float(t)),(510*i,0))
            frames.append(img)
    finally:
        for cap in caps:cap.release()
    out=RUN_DIR/f"Good_vs_{bad['case']}.gif"
    frames[0].save(out,save_all=True,append_images=frames[1:],duration=round(1000/GIF_FPS),loop=0)
    display(DisplayImage(filename=str(out)));print(out)
    if full_duration>duration:print(f'GIF는 앞 {duration:.0f}초 미리보기입니다. 전체 결과는 JSON/CSV에서 확인하세요.')


## 9. 그래프·비교 요약·설정 저장
합쳐진 무릎/횟수 그래프와 케이스별 좌우 무릎·골반·팔꿈치 각도 6개 그래프입니다. 일정한 그래프를 정자세로 간주하지 않습니다. 결과 폴더는 실행마다 새로 만들어집니다.


In [ ]:
rows=[]
fig,axes=plt.subplots(2,1,figsize=(12,8),constrained_layout=True)
for c in CLIPS:
    d=DATA[c['path']];table=d['table'].iloc[c['ids']];t=table.time_s-c['render_start']
    if 'kneeDeg' in table:axes[0].plot(t,table.kneeDeg,label=c['case'])
    axes[1].step(t,c['counts'][c['ids']],where='post',label=c['case'])
    segment=table.copy();segment['segmentCount']=c['counts'][c['ids']];segment.to_csv(RUN_DIR/(c['case']+'_segment.csv'),index=False,encoding='utf-8-sig')
    rows.append(dict(case=c['case'],label=c['label'],labelSource=c['labelSource'],source=Path(c['path']).name,segmentCount=int(c['counts'][c['ids'][-1]]),sourceCount=d['completed']['summary']['count'],feedback=' / '.join(c['summaryFeedback'])))
    f,axs=plt.subplots(3,2,figsize=(12,10),constrained_layout=True)
    for row,key in enumerate(['kneeDeg','hipDeg','elbowDeg']):
        for col,s in enumerate(SIDES):
            ax=axs[row,col];ax.plot(t,table[s+'_'+key]);ax.set_title(s+' '+key);ax.set_xlabel('Elapsed seconds');ax.set_ylabel('Degrees');ax.grid(alpha=.2)
    f.suptitle(c['case']);f.savefig(RUN_DIR/(c['case']+'_six_angles.png'),dpi=140);plt.show()
for ax,title in zip(axes,['Selected-side knee angle','Detected segment repetitions']):ax.set_title(title);ax.set_xlabel('Elapsed seconds');ax.legend();ax.grid(alpha=.2)
axes[0].axhline({**DEFAULTS[EXERCISE],**THRESHOLD_OVERRIDES}['depth'],linestyle='--',color='gray',label='Experimental depth threshold')
fig.savefig(RUN_DIR/'combined_metrics.png',dpi=150);plt.show()
summary_table=pd.DataFrame(rows);display(summary_table)
summary_table.to_csv(RUN_DIR/'comparison_summary.csv',index=False,encoding='utf-8-sig')
(RUN_DIR/'settings.json').write_text(json.dumps(dict(exercise=EXERCISE,engineVersion=BarofitEngine.VERSION,side=SIDE,overrides=THRESHOLD_OVERRIDES,sideOverrides=SIDE_OVERRIDES,sampleFPS=SAMPLE_FPS,experimental=True,contract='proposed squat/lunge v2 extension',limitations=['측면·단독 인물 전제','척추 중립·무릎 내측 붕괴·통증 여부 판정 안함','125도에 미달하는 얕은 시도는 미집계','원본 라벨과 규칙 결과는 독립']),ensure_ascii=False,indent=2),encoding='utf-8')
print('저장 완료:',RUN_DIR)


## 10. 합성 신호·반환 규격 점검
실제 영상 정확도 검증과는 별개입니다. 실제 영상의 카운트·측면 분석측·관절 가림·시작 경계를 눈으로 대조해야 합니다. 동일 로직의 웹/모바일 이식은 별도 작업입니다.


In [ ]:
def run_signal(exercise,values,trunk=15.,side='left'):
    e=BarofitEngine(exercise,side);out=[]
    for i,k in enumerate(values):
        raw={'left':dict(kneeDeg=k,hipDeg=130.,elbowDeg=170.,trunkDeg=trunk)}
        r=e.step(i*100,raw);validate_result(r);out.append(r)
    summary=e.finish();validate_result(summary)
    return e,out,summary
cycle=[175.]*15+list(np.linspace(175,95,20))+[95.]*10+list(np.linspace(95,175,20))+[175.]*15
shallow=[175.]*15+list(np.linspace(175,120,20))+[120.]*10+list(np.linspace(120,175,20))+[175.]*15
for exercise in ['squat','lunge']:
    e,out,s=run_signal(exercise,cycle*2)
    assert e.count==2 and s['summary']['items']['depth']['compliant']==2
    e,out,s=run_signal(exercise,shallow,trunk=55.)
    assert e.count==1 and s['summary']['items']['depth']['warning']==1
    assert s['summary']['items']['trunk']['warning']==1
    assert not any(r['currentCodes'] and 'DEPTH_SHALLOW' in r['currentCodes'] for r in out)
    assert any(r['lastRepetition'] and 'DEPTH_SHALLOW' in r['lastRepetition']['codes'] for r in out)
    e,out,s=run_signal(exercise,[175.]*100);assert e.count==0 and s['summary']['items']['depth']['compliancePct'] is None
    e,out,s=run_signal(exercise,[95.]*15+[175.]*20);assert e.count==0
    e,out,s=run_signal(exercise,cycle,trunk=None);assert e.count==0
    e=BarofitEngine(exercise,'left');raw={'left':dict(kneeDeg=175.,trunkDeg=15.)}
    e.step(100,raw);before=copy.deepcopy(e.__dict__)
    ignored=e.step(100,raw);validate_result(ignored);assert ignored['ignoredReason']=='non_monotonic_timestamp' and before==e.__dict__
    finish=e.finish();assert finish==e.finish()
    ignored=e.step(200,raw);validate_result(ignored);assert ignored['ignoredReason']=='finished'
# 인식 중단과 큰 시간 간격은 완료 횟수를 올리지 않습니다.
e=BarofitEngine('squat','left')
for i,k in enumerate([175.]*15+[95.]*15):e.step(i*100,{'left':dict(kneeDeg=k,trunkDeg=15.)})
e.step(5000,{'left':dict(kneeDeg=None,trunkDeg=None)})
for i in range(20):e.step(5100+i*100,{'left':dict(kneeDeg=175.,trunkDeg=15.)})
assert e.count==0 and e.interrupted>=1
print('PASS: 스키마·2회 계수·얕은 동작 경고·현재/회차 분리·정지·시작 경계·중단·역순 입력·종료 멱등성')
